# Feature engineering

Data Scientist track, lesson DS9. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** fill and flag missing values inside the folds, encode text columns to match the model, apply log, ratio and date-part transforms and judge each by its paired gain, and see how a column built from the labels fakes a gain.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder

SEED = 2026       # the seed of the whole Data Scientist track

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)

# The modeling table of the Data Scientist track: one row per person, built by the query of lesson DS3, "Building a modeling table in SQL".
# Run this cell as it is; DS3 explains it line by line.
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"Data loaded: {len(table):,} people, {int(table['churned'].sum()):,} of them churned.")

## The split, the new columns and the recipe

The columns and the split of lessons DS2 and DS4 (the test part stays untouched until the last unit) and the five folds of lesson DS5. The table `T` gets four columns built from what it already holds: a ratio that is undefined for some people, two date parts, and a flag. `recipe` is the day-7 model with options for each unit. Run this cell as it is.

In [ ]:
STATIC = ["channel", "platform", "region"]                    # known at signup
WEEK1 = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d", "onboarded", "saw_paywall", "bought_plus"]    # known at day 7
X, y = table[STATIC + WEEK1], table["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)       # 13,362 and 4,454 people
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train)   # 10,021 and 3,341, cut from the training part
folds = StratifiedKFold(5, shuffle=True, random_state=SEED)       # the five folds of lesson DS5


def logit():
    """The day-7 model of lesson DS2: one-hot text columns, scaled number columns, logistic regression."""
    return make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", StandardScaler(), WEEK1)]), LogisticRegression(max_iter=1000))


def auc_gain(p_new, p_base, y_true, resamples=2000):
    """ROC AUC of one model minus another on the same people, with a 95 percent bootstrap interval; both models are scored on the same resamples."""
    rng = np.random.default_rng(SEED)
    y_true, a, b = np.asarray(y_true), np.asarray(p_new), np.asarray(p_base)
    gains = []
    for _ in range(resamples):
        i = rng.integers(0, len(y_true), len(y_true))
        gains.append(roc_auc_score(y_true[i], a[i]) - roc_auc_score(y_true[i], b[i]))
    low, high = np.percentile(gains, [2.5, 97.5])
    return [round(float(roc_auc_score(y_true, a) - roc_auc_score(y_true, b)), 3) + 0.0, round(float(low), 3) + 0.0, round(float(high), 3) + 0.0]

T = table.copy()
T["signup_date"] = pd.to_datetime(T["signup_date"])
T["expenses_per_day"] = T["expenses_7d"] / T["active_days_7d"].where(T["active_days_7d"] > 0)        # expenses on each active day; undefined (missing) when there was no active day
T["weekday"] = T["signup_date"].dt.day_name()                                                          # a date part: the day of the week of the signup
T["days_since_jan1"] = (T["signup_date"] - pd.Timestamp("2026-01-01")).dt.days                          # a date part: days from the first day of the data to the signup
T["no_active_day"] = (T["active_days_7d"] == 0).astype(int)                                             # a flag
Xt, Xs = T.loc[X_train.index], T.loc[X_test.index]                                                      # the training and test people with every column
COUNTS = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d"]


def recipe(cat=STATIC, num=WEEK1, log=(), fill=False, flag=False, ordinal=False):
    """The day-7 recipe with options: extra columns, counts on a log scale, a median fill with or without a 0/1 flag, ordinal codes instead of one-hot columns."""
    encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1) if ordinal else OneHotEncoder(handle_unknown="ignore")
    steps = ([SimpleImputer(strategy="median", add_indicator=flag)] if fill else []) + [StandardScaler()]
    parts = [("cat", encoder, list(cat)), ("num", make_pipeline(*steps), [c for c in num if c not in log])]
    if log:
        parts.append(("log", make_pipeline(FunctionTransformer(np.log1p), StandardScaler()), list(log)))
    return make_pipeline(ColumnTransformer(parts), LogisticRegression(max_iter=2000))


def folds_auc(model, cols):
    """ROC AUC of a model on each of the five folds of the training part."""
    return cross_val_score(model, Xt[cols], y_train, cv=folds, scoring="roc_auc")


def gain(scores, base):
    """The mean of the five fold-by-fold differences from the base, and their lowest and highest."""
    d = scores - base
    return [round(float(d.mean()), 4) + 0.0, round(float(d.min()), 4) + 0.0, round(float(d.max()), 4) + 0.0]


BASE_COLS = STATIC + WEEK1
base_scores = folds_auc(recipe(), BASE_COLS)
print(len(Xt), len(Xs), round(float(base_scores.mean()), 4))

## Unit 1: Fill a gap, and flag it

**Predict first.** How many of the training people will have no value, and will they churn at a different rate? And will filling the gap, with or without the flag, lift the mean AUC by more than 0.005?

In [ ]:
missing = Xt["expenses_per_day"].isna()
print(round(float(missing.mean()), 3), y_train.groupby(missing).mean().round(3).to_dict())          # the share without a value, and the churn rate of those with and without
COLS_R = BASE_COLS + ["expenses_per_day"]
for tag, options in [("median fill", dict(fill=True)), ("median fill and flag", dict(fill=True, flag=True))]:
    s = folds_auc(recipe(num=WEEK1 + ["expenses_per_day"], **options), COLS_R)
    print(f"{tag:22s} mean {s.mean():.4f}   gain over the day-7 model {gain(s, base_scores)}")
print("the day-7 model:", round(float(base_scores.mean()), 4))

## Unit 2: Encode to match the model

**Predict first.** With the three text columns written as codes, will a logistic regression lose about 0.001, about 0.03 or about 0.1 AUC? And a random forest?

In [ ]:
def forest_scores(ordinal):
    encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1) if ordinal else OneHotEncoder(handle_unknown="ignore")
    m = make_pipeline(ColumnTransformer([("cat", encoder, STATIC), ("num", "passthrough", WEEK1)]), RandomForestClassifier(200, min_samples_leaf=100, n_jobs=-1, random_state=SEED))
    return folds_auc(m, BASE_COLS)


enc_rows = []
for tag, ordinal in [("one-hot columns", False), ("codes 0, 1, 2, ...", True)]:
    enc_rows.append((tag, round(float(folds_auc(recipe(ordinal=ordinal), BASE_COLS).mean()), 3), round(float(forest_scores(ordinal).mean()), 3)))
print(pd.DataFrame(enc_rows, columns=["the text columns as", "logistic regression", "random forest"]).to_string(index=False))

## Unit 3: Transforms, judged by paired gains

**Predict first.** Four changes: the four counts on a log scale, the weekday of signup, the days since January 1, and a flag for no active day. Will any of them gain more than the spread of its own fold differences?

In [ ]:
menu = {"counts on a log scale": (recipe(log=COUNTS), BASE_COLS),
        "weekday of signup, one-hot": (recipe(cat=STATIC + ["weekday"]), BASE_COLS + ["weekday"]),
        "days since January 1": (recipe(num=WEEK1 + ["days_since_jan1"]), BASE_COLS + ["days_since_jan1"]),
        "flag: no active day": (recipe(num=WEEK1 + ["no_active_day"]), BASE_COLS + ["no_active_day"])}
menu_gains = {}
for tag, (model, cols) in menu.items():
    menu_gains[tag] = gain(folds_auc(model, cols), base_scores)
    print(f"{tag:30s} gain {menu_gains[tag][0]:+.4f}   fold differences from {menu_gains[tag][1]:+.4f} to {menu_gains[tag][2]:+.4f}")

## Unit 4: A column made from the labels

**Predict first.** What will the two routes report on the folds: near the day-7 model's 0.692, above 0.70, or below 0.67?

In [ ]:
T["cohort"] = T["signup_date"].dt.strftime("%m-%d") + "_" + T["channel"]       # the day and the channel of the signup
Xt = T.loc[X_train.index]
print(Xt["cohort"].nunique(), "groups, the middle one with", int(Xt["cohort"].value_counts().median()), "people")
shares = y_train.groupby(Xt["cohort"]).mean()                                  # the share of churners in each group, from every training person
outside = Xt[BASE_COLS].assign(cohort_share=Xt["cohort"].map(shares))
leaky = cross_val_score(recipe(num=WEEK1 + ["cohort_share"]), outside, y_train, cv=folds, scoring="roc_auc")


class GroupShare(BaseEstimator, TransformerMixin):
    """The share of churners in each group, learned in fit() from the fitting people only; a group it has not seen gets the overall share."""

    def fit(self, X, y):
        self.shares_ = pd.Series(np.asarray(y)).groupby(np.asarray(X).ravel()).mean()
        self.overall_ = float(np.mean(y))
        return self

    def transform(self, X):
        return pd.Series(np.asarray(X).ravel()).map(self.shares_).fillna(self.overall_).to_numpy().reshape(-1, 1)


inside_model = make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", StandardScaler(), WEEK1),
                                                ("share", make_pipeline(GroupShare(), StandardScaler()), ["cohort"])]), LogisticRegression(max_iter=2000))
honest = folds_auc(inside_model, BASE_COLS + ["cohort"])
print("outside the folds", round(float(leaky.mean()), 3), "inside the folds", round(float(honest.mean()), 3), "the day-7 model", round(float(base_scores.mean()), 3))

## Unit 5: The verdict on the test part

**Predict first.** Will the recipe with the new column beat the day-7 model by more than 0.005 AUC?

In [ ]:
p_day7 = recipe().fit(Xt[BASE_COLS], y_train).predict_proba(Xs[BASE_COLS])[:, 1]                       # the test part is used once, here
final = recipe(num=WEEK1 + ["expenses_per_day"], fill=True, flag=True).fit(Xt[COLS_R], y_train)
p_final = final.predict_proba(Xs[COLS_R])[:, 1]
print("day-7 model", round(float(roc_auc_score(y_test, p_day7)), 3), "with the ratio, fill and flag", round(float(roc_auc_score(y_test, p_final)), 3))
print(auc_gain(p_final, p_day7, y_test))

## Finish it

### Exercise 1: Finish it

**Task:** **Do the changes add up?** Build `recipe` with `num=WEEK1 + ["expenses_per_day"]`, the four `COUNTS` on a log scale (`log=COUNTS`) and the median fill with a flag, score it with `folds_auc(model, cols_stack)`, and put the list that `gain(scores, base_scores)` returns in `answer_1`.

**Expected:** `answer_1` is a list of three numbers.

In [ ]:
cols_stack = BASE_COLS + ["expenses_per_day"]          # given: the day-7 columns and the ratio
# build recipe(...) with num=WEEK1 + ["expenses_per_day"], the four COUNTS on a log scale (log=COUNTS) and the median fill with a flag,
# score it on the five folds with folds_auc(model, cols_stack), and take gain(scores, base_scores)
answer_1 = None  # replace None with the list that gain(...) returns

In [ ]:
_ = check("Exercise 1", "answer_1", [0.0003, -0.0014, 0.0016], traps=[([0.0007, -0.0011, 0.0012], "That is the gain of the ratio alone, from unit 1. Add log=COUNTS as well.")], tol=1e-4)

<details>
<summary>Hint</summary>

`recipe(num=WEEK1 + ["expenses_per_day"], log=COUNTS, fill=True, flag=True)` is the model; `folds_auc(model, cols_stack)` gives its five fold scores.

</details>

<details>
<summary>Half-solution</summary>

```python
stack = folds_auc(recipe(num=WEEK1 + ["expenses_per_day"], log=..., fill=True, flag=True), cols_stack)
answer_1 = gain(stack, base_scores)
```

</details>

<details>
<summary>Solution</summary>

```python
stack = folds_auc(recipe(num=WEEK1 + ["expenses_per_day"], log=COUNTS, fill=True, flag=True), cols_stack)
answer_1 = gain(stack, base_scores)
```

The two changes together gain 0.0003, with five differences from -0.0014 to 0.0016. Two changes that add nothing alone add nothing together, and the spread is as wide as for either one.

</details>

## On your own

### Exercise 2: On your own

**Task:** **Another ratio.** Before you start, say in one sentence what the result looks like: a mean gain near zero and a range of fold differences that includes it. Add `visit_share`, the sessions divided by the sessions plus the expenses, which is undefined when both are 0, to `Xt` (the training people with every column). Score the day-7 columns plus `visit_share` with the median fill and the flag on the five folds, and put the list that `gain(scores, base_scores)` returns in `answer_2`. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of three numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [0.0004, -0.0012, 0.0011], traps=[([0.0007, -0.0011, 0.0012], "That is the ratio of unit 1: expenses per active day. Build visit_share from sessions and expenses.")], tol=1e-4)

<details>
<summary>Hint</summary>

Make the column with `Xt["sessions_7d"] / (Xt["sessions_7d"] + Xt["expenses_7d"]).where(Xt["sessions_7d"] + Xt["expenses_7d"] > 0)`, then call `recipe(num=WEEK1 + ["visit_share"], fill=True, flag=True)` on the columns `BASE_COLS + ["visit_share"]`.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"Three numbers, the first within a thousandth of zero, the other two on either side of it."*

```python
Xt["visit_share"] = Xt["sessions_7d"] / (Xt["sessions_7d"] + Xt["expenses_7d"]).where(Xt["sessions_7d"] + Xt["expenses_7d"] > 0)
s_visit = folds_auc(recipe(num=WEEK1 + ["visit_share"], ...), BASE_COLS + ["visit_share"])
```

</details>

<details>
<summary>Solution</summary>

```python
Xt["visit_share"] = (Xt["sessions_7d"] / (Xt["sessions_7d"] + Xt["expenses_7d"]).where(Xt["sessions_7d"] + Xt["expenses_7d"] > 0))
s_visit = folds_auc(recipe(num=WEEK1 + ["visit_share"], fill=True, flag=True), BASE_COLS + ["visit_share"])
answer_2 = gain(s_visit, base_scores)
```

The gain is 0.0004, with differences from -0.0012 to 0.0011. People with no session and no expense have no value, the same 0.250 as in unit 1, so the new column is a second view of the same information.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** A colleague fills the ratio with its median in pandas, then makes the flag for "had no value" from the filled column, and says the flag sums to a count of people (the code is given). Look at what `answer_3` holds, then change the line that makes the flag so that `answer_3` counts the training people who had no value.

**Expected:** `answer_3` is a whole number above 1,000.

In [ ]:
ratio = Xt["expenses_per_day"]
filled = ratio.fillna(ratio.median())            # the fill, done in pandas
flagged = filled.isna().astype(int)              # the flag for "had no value"
answer_3 = int(flagged.sum())                    # colleague: "the flag marks the people who had no value; it sums to this many"
# change the line that makes the flag, so that answer_3 counts the training people who had no value

In [ ]:
_ = check("Exercise 3", "answer_3", 3336, traps={0: "That is the flag made after the fill: nothing is missing any more, so it never fires. Make the flag from the column as it was."})

<details>
<summary>Hint</summary>

After `fillna` the column has no missing value left. Which column should `isna()` read?

</details>

<details>
<summary>Half-solution</summary>

```python
flagged = ratio.isna().astype(int)
answer_3 = int(flagged.sum())
```

</details>

<details>
<summary>Solution</summary>

```python
flagged = ratio.isna().astype(int)
answer_3 = int(flagged.sum())
```

The flag made from the filled column sums to 0: the fill had already replaced every gap, so the model would get a column of zeros, and nothing raises an error. Made from the original column it counts 3,336 training people, the 25.0 percent of unit 1. Inside a pipeline `add_indicator=True` makes the flag before it fills, which is why a pipeline is safer than two pandas lines.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")